# Info-dict wrappers

> Put everything about a step in one info dict, like stable-worldmodel's `MegaWrapper`.

In [ ]:
#| default_exp wrappers.default

In [ ]:
#| hide
from nbdev.showdoc import *

The multi-agent counterparts of stable-worldmodel's default wrappers. A wrapped env still returns
``(observations, rewards, terminations, truncations, info)`` with per-agent dicts, but its
`info` is one flat dict holding everything a policy or a dataset needs:

| key | shape | content |
|---|---|---|
| observation keys (`image`, `pov`, `direction`, ...) | `(num_agents, ...)` | each agent's observation |
| the env's own per-agent info keys | `(num_agents, ...)` | |
| `reward`, `terminated`, `truncated` | `(num_agents,)` | of the step that led here (`nan`, False after a reset) |
| `action` | `(num_agents, *action_shape)` | the action that led here (`nan` after a reset); for agents that had already terminated, the env's `noop_action` (`nan` if it has none) |
| `state` | env-specific | global state (`env.state()`) |
| `step_idx`, `id`, `seed` | `()` | step in the episode, episode id, reset seed (-1: none) |
| `goal`, `goal_position` | `(num_agents, ...)` | each agent's goal observation and cell (envs with goals) |
| `variation.<name>` | factor-specific | value of a watched factor of variation (see below) |
| `pixels` | `(H, W, 3)` | render of the whole env (`AddPixelsWrapper`) |

As in stable-worldmodel, the factors of variation named in a reset's ``options['variation']``
(every factor for ``['all']``) are *watched* from then on: their values are added to every info,
so datasets record them. Factors set with ``options['variation_values']`` are watched too.

`MegaWrapper` chains them; `World` wraps every env with it, and `EnvPool` stacks the infos.

In [ ]:
#| export
from __future__ import annotations

import re
import time
from typing import Any, Callable, Iterable

import gymnasium as gym
import numpy as np

from stable_marl.spaces import get_in

In [ ]:
#| export
def _is_array_like(value) -> bool:
    return isinstance(value, (np.ndarray, np.generic, bool, int, float)) and not isinstance(value, str)


def _stack_agents(values: list) -> Any:
    "The values of all agents, stacked (a list if they are not arrays)."
    return np.stack([np.asarray(v) for v in values]) if all(_is_array_like(v) for v in values) else list(values)


class EverythingToInfoWrapper(gym.Wrapper):
    "Moves everything about a step into one flat info dict (see the table above)."

    def __init__(self, env: gym.Env):
        super().__init__(env)
        self._step_counter, self._id, self._seed = 0, 0, -1
        self._variations_watch: list[str] = []
        self._finished = None   # agents terminated before the current step

    @property
    def num_agents(self) -> int:
        return self.env.unwrapped.num_agents

    def _info(self, obs: dict, rewards, terminations, truncations, actions, env_infos: dict) -> dict:
        A = self.num_agents
        info = {key: _stack_agents([obs[a][key] for a in range(A)]) for key in obs[0]}
        for key in (env_infos.get(0) or {}):
            assert key not in info, f"info key {key!r} is also an observation key"
            info[key] = _stack_agents([env_infos[a].get(key) for a in range(A)])
        for key, value in (('reward', np.array([rewards[a] for a in range(A)], np.float32)),
                           ('terminated', np.array([terminations[a] for a in range(A)], bool)),
                           ('truncated', np.array([truncations[a] for a in range(A)], bool)),
                           ('action', np.stack([np.asarray(actions[a], np.float32) for a in range(A)])),
                           ('state', self.env.unwrapped.state()),
                           ('step_idx', np.int64(self._step_counter)),
                           ('id', np.int64(self._id)),
                           ('seed', np.int64(self._seed))):
            assert key not in info, f"{key!r} is reserved, the env cannot use it as an observation or info key"
            info[key] = value
        space = getattr(self.env.unwrapped, 'variation_space', None)
        for name in self._variations_watch:
            info[f'variation.{name}'] = np.asarray(get_in(space, name.split('.')).value)
        return info

    def _watch_variations(self, options: dict | None):
        "Watch the factors named in the reset options, expanded to leaf factors."
        options = options or {}
        if 'variation' not in options and 'variation_values' not in options:
            return
        space = getattr(self.env.unwrapped, 'variation_space', None)
        if space is None:
            raise ValueError(f"{type(self.env.unwrapped).__name__} has no variation_space")
        requested = list(options.get('variation', ())) + list(options.get('variation_values', {}))
        names = space.names()
        watch = names if 'all' in requested else [n for n in names if any(n == r or n.startswith(r + '.') for r in requested)]
        self._variations_watch = list(dict.fromkeys(self._variations_watch + watch))

    def reset(self, *, seed: int | None = None, options: dict | None = None):
        obs, env_infos = self.env.reset(seed=seed, options=options)
        self._watch_variations(options)
        self._step_counter, self._seed = 0, -1 if seed is None else int(seed)
        # episode id: from the seed (reproducible), without touching the env's own RNG
        rng = np.random.default_rng(None if seed is None else [int(seed), 0x5EED])
        self._id = int(rng.integers(np.iinfo(np.int64).max))
        A = self.num_agents
        self._finished = np.zeros(A, bool)
        return obs, self._info(obs, [np.nan] * A, [False] * A, [False] * A, self._no_action(), env_infos)

    def _no_action(self) -> list[np.ndarray]:
        return [np.full(self.env.action_space[a].shape, np.nan, np.float32) for a in range(self.num_agents)]

    def step(self, actions: dict):
        # agents that already terminated take the env's no-op (recorded as nan if it has none)
        noop = getattr(self.env.unwrapped, 'noop_action', None)
        finished = self._finished if self._finished is not None else np.zeros(self.num_agents, bool)
        actions = dict(actions)
        if noop is not None:
            actions.update({int(a): noop for a in np.flatnonzero(finished)})
        obs, rewards, terminations, truncations, env_infos = self.env.step(actions)
        self._step_counter += 1
        no_action = self._no_action()
        acts = [no_action[a] if finished[a] and noop is None else np.asarray(actions[a]) for a in range(self.num_agents)]
        self._finished = np.array([terminations[a] for a in range(self.num_agents)], bool)
        info = self._info(obs, rewards, terminations, truncations, acts, env_infos)
        return obs, rewards, terminations, truncations, info

In [ ]:
#| export
_RESAMPLE = {'nearest': 'NEAREST', 'bilinear': 'BILINEAR', 'bicubic': 'BICUBIC',
             'lanczos': 'LANCZOS', 'box': 'BOX', 'hamming': 'HAMMING'}


def _resolve_resample(resample: str | int | None) -> int | None:
    if resample is None or isinstance(resample, int):
        return resample
    from PIL import Image
    if resample not in _RESAMPLE:
        raise ValueError(f"image_resample must be one of {sorted(_RESAMPLE)}, got {resample!r}")
    return getattr(Image.Resampling, _RESAMPLE[resample])


class AddPixelsWrapper(gym.Wrapper):
    """
    Adds a render of the whole env (``env.render()``, so ``render_mode='rgb_array'``) to the
    info as ``pixels``, resized to `image_shape` (H, W) if given, then passed through
    `transform`. Also adds ``render_time`` (seconds).
    """
    def __init__(
        self,
        env: gym.Env,
        image_shape: tuple[int, int] | None = None,
        transform: Callable[[np.ndarray], Any] | None = None,
        resample: str | int | None = None):
        super().__init__(env)
        if env.render_mode != 'rgb_array':
            raise ValueError("AddPixelsWrapper needs an env made with render_mode='rgb_array'")
        self.image_shape, self.transform = image_shape, transform
        self.resample = _resolve_resample(resample)

    def _add_pixels(self, info: dict) -> dict:
        start = time.time()
        img = self.env.render()
        if self.image_shape is not None and img.shape[:2] != tuple(self.image_shape):
            from PIL import Image
            h, w = self.image_shape
            resample = Image.Resampling.BILINEAR if self.resample is None else self.resample
            img = np.asarray(Image.fromarray(img).resize((w, h), resample=resample))
        info['pixels'] = self.transform(img) if self.transform else img
        info['render_time'] = time.time() - start
        return info

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        return obs, self._add_pixels(info)

    def step(self, actions):
        obs, rewards, terminations, truncations, info = self.env.step(actions)
        return obs, rewards, terminations, truncations, self._add_pixels(info)

In [ ]:
#| export
class EnsureInfoKeysWrapper(gym.Wrapper):
    "Checks that the info of every reset and step has the `required_keys` (regular expressions)."

    def __init__(self, env: gym.Env, required_keys: Iterable[str]):
        super().__init__(env)
        self._patterns = [re.compile(key) for key in required_keys]

    def _check(self, info: dict, where: str):
        for pattern in self._patterns:
            if not any(pattern.fullmatch(key) for key in info):
                raise RuntimeError(f"{where}: no info key matches the required {pattern.pattern!r} "
                                   f"(keys: {sorted(info)})")

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._check(info, 'reset')
        return obs, info

    def step(self, actions):
        obs, rewards, terminations, truncations, info = self.env.step(actions)
        self._check(info, 'step')
        return obs, rewards, terminations, truncations, info

In [ ]:
#| export
class TransformInfoWrapper(gym.Wrapper):
    "Applies `transform` to ``info[key]`` (when present) after every reset and step."

    def __init__(self, env: gym.Env, key: str, transform: Callable[[Any], Any]):
        super().__init__(env)
        self.key, self.transform = key, transform

    def _apply(self, info: dict) -> dict:
        if self.key in info:
            info[self.key] = self.transform(info[self.key])
        return info

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        return obs, self._apply(info)

    def step(self, actions):
        obs, rewards, terminations, truncations, info = self.env.step(actions)
        return obs, rewards, terminations, truncations, self._apply(info)


class MegaWrapper(gym.Wrapper):
    """
    The standard preprocessing of every env of a :class:`World`: everything goes to the info dict
    (:class:`EverythingToInfoWrapper`), plus the render as ``pixels`` if `add_pixels`
    (:class:`AddPixelsWrapper`), and a check of the `required_keys`.

    Parameters
    ----------
    image_shape : (H, W), optional
        Size ``pixels`` is resized to (default: the render's size)
    pixels_transform : callable, optional
        Applied to ``pixels``
    required_keys : list of str, optional
        Info keys (regular expressions) that must be present
    image_resample : str or int, optional
        PIL resampling used to resize ``pixels`` ('nearest', 'bilinear', ...), default bilinear
    add_pixels : bool
        Render the env into ``pixels`` (needs ``render_mode='rgb_array'``)
    goal_transform : callable, optional
        Applied to ``goal`` (the agents' goal observations)
    separate_goal : bool
        Require a ``goal`` in the info (goal-conditioned envs)
    """
    def __init__(
        self,
        env: gym.Env,
        image_shape: tuple[int, int] | None = None,
        pixels_transform: Callable[[np.ndarray], Any] | None = None,
        required_keys: Iterable[str] | None = None,
        image_resample: str | int | None = None,
        add_pixels: bool = False,
        goal_transform: Callable[[Any], Any] | None = None,
        separate_goal: bool = False):
        super().__init__(env)
        required = list(required_keys or [])
        env = EverythingToInfoWrapper(env)
        if add_pixels:
            required.append(r'^pixels(?:\..*)?$')
            env = AddPixelsWrapper(env, image_shape, pixels_transform, image_resample)
        if separate_goal:
            required.append('goal')
        if goal_transform is not None:
            env = TransformInfoWrapper(env, 'goal', goal_transform)
        self.env = EnsureInfoKeysWrapper(env, required)

### Tests

In [ ]:
from fastcore.test import test_fail
from stable_marl.envs import make

env = MegaWrapper(make('MultiGrid-Empty-6x6-v0', agents=2, render_mode='rgb_array'), add_pixels=True, image_shape=(64, 48))
obs, info = env.reset(seed=3)
assert info['pov'].shape == (2, 224, 224, 3) and info['image'].shape == (2, 7, 7, 3)
assert info['pixels'].shape == (64, 48, 3) and info['state'].shape == (6, 6, 3) and info['seed'] == 3
assert np.isnan(info['reward']).all() and np.isnan(info['action']).all() and info['step_idx'] == 0
assert np.array_equal(info['pov'][1], obs[1]['pov'])          # observations are still returned as usual
obs, rew, term, trunc, info = env.step({0: 2, 1: 1})
assert list(info['action']) == [2, 1] and info['step_idx'] == 1 and info['reward'][0] == rew[0]
assert np.array_equal(info['state'], env.unwrapped.state())

# agents that already terminated take (and record) the env's no-op, the others their own action
env = MegaWrapper(make('MultiGrid-FindGoal-15x15-v0', agents=2, size=7, num_obstacles=0, n_clutter=0))
inner = env
while not isinstance(inner, EverythingToInfoWrapper): inner = inner.env
env.reset(seed=0)
assert env.unwrapped.noop_action == 3                         # FindGoal's `done`
env.unwrapped.agents[0].state.terminated, inner._finished = True, np.array([True, False])   # agent 0 has finished
_, _, _, _, info = env.step({0: 2, 1: 1})
assert list(info['action']) == [3, 1]
cls = type(env.unwrapped)
cls.noop_action = None                                        # an env without a no-op: nan
try:
    _, _, _, _, info = env.step({0: 2, 1: 1})
    assert np.isnan(info['action'][0]) and info['action'][1] == 1
finally:
    del cls.noop_action
assert env.unwrapped.noop_action == 3

# the episode id comes from the seed, without changing the env's randomness
first_id = MegaWrapper(make('MultiGrid-Empty-6x6-v0', agents=2)).reset(seed=3)[1]['id']
assert MegaWrapper(make('MultiGrid-Empty-6x6-v0', agents=2)).reset(seed=3)[1]['id'] == first_id
raw = make('MultiGrid-Empty-6x6-v0', agents=2)
assert np.array_equal(raw.reset(seed=3)[0][0]['pov'], MegaWrapper(make('MultiGrid-Empty-6x6-v0', agents=2)).reset(seed=3)[0][0]['pov'])

test_fail(lambda: MegaWrapper(make('MultiGrid-RedBlueDoors-6x6-v0'), separate_goal=True).reset(), contains="'goal'")

# goals and watched variations
env = MegaWrapper(make('MultiGrid-FindGoal-15x15-v0', agents=2, num_obstacles=6), goal_transform=lambda g: g[..., ::2, ::2, :])
_, info = env.reset(seed=0, options={'variation': ['wall.color', 'obstacles']})
assert info['goal'].shape == (2, 112, 112, 3) and info['goal_position'].shape == (2, 2)
assert {'variation.wall.color', 'variation.obstacles.number'} <= set(info) and 'variation.agent.color' not in info
_, _, _, _, info = env.step({0: 2, 1: 2})
assert 'variation.wall.color' in info and info['goal'].shape == (2, 112, 112, 3)
_, info = env.reset(seed=1, options={'variation': ['all']})
assert {f'variation.{n}' for n in env.unwrapped.variation_space.names()} <= set(info)
test_fail(lambda: MegaWrapper(make('MultiGrid-Empty-6x6-v0'), add_pixels=True), contains='rgb_array')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()